# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kiran162005/flyrank-ml/blob/main/work/notebooks/w05_model.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*


### Method choice: Logistic Regression

I will use **Logistic Regression** for the Refresh / Content Opportunity Scoring lane.

The goal is to identify pages that may deserve content review using observable search-performance signals. Logistic Regression fits this task because it predicts the probability of a binary outcome while remaining relatively simple and interpretable.

The model will use signals that are available at the decision moment, such as search impressions, CTR, and average search position. Because search-performance variables have heavy-tailed distributions, I will use appropriate transformations such as `log1p(impressions)` where needed.

I chose Logistic Regression as the first model because the purpose of this stage is to test whether a simple statistical model can provide useful improvement over the Week-4 rule-based baseline. I do not want to reward model complexity unless it produces a meaningful improvement on the same evaluation data.

The target will represent a **later observed outcome**, while the model features will come only from information available before that outcome. This avoids using future performance as an input and makes the model a genuine prediction rather than a reproduction of the Week-4 rule.

The model will be evaluated against the Week-4 baseline using the same held-out observations and the same evaluation metric. I will also inspect errors and feature effects to determine whether the model is useful for decision-support.

The model does not establish that refreshing a page will cause its performance to improve. It is intended to identify pages that deserve further human review.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*


### Time-aware and client-grouped validation

I will use a **time-aware split** because the modeling question is whether information available at an earlier decision point can help identify a later content-performance outcome.

The model will be trained using observations from the earlier period and evaluated on a later held-out period. This prevents future information from being used to predict the past.

I will also keep observations from the same `client_hash_id` together when constructing the validation split where possible. This reduces the risk that the model learns client-specific patterns from the training data and then receives very similar observations from the same client during evaluation.

The **Week-4 baseline and the Logistic Regression model will be evaluated on the same held-out observations using the same metric**. This makes the comparison fair.

I will not randomly mix future and earlier observations because that could create temporal leakage. I will also avoid using future-period performance, future labels, or product-generated flags as model features.

### Why this split is honest

The intended use is decision-support: given what was known at the decision moment, estimate which pages may deserve attention later. A time-aware evaluation therefore better represents how the model would behave in practice.

The held-out period is treated as unseen data. Performance on this period is used to judge whether the model actually improves on the Week-4 baseline rather than simply fitting the development data.


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*


I will train Logistic Regression using only features available at the decision point and compare it with the Week-4 rule-based baseline on the **same held-out observations and the same evaluation metric**.

The target represents a later observed outcome. The model does not use future performance as an input.

The comparison focuses on whether the model provides better decision-support than the transparent Week-4 rule, rather than whether the model is more complex.


In [2]:
import duckdb
from google.colab import userdata

# Get Hugging Face token
HF_TOKEN = userdata.get("HF_TOKEN")

if not HF_TOKEN:
    raise ValueError(
        "HF_TOKEN is missing. Add your Hugging Face token "
        "to Colab Secrets."
    )

print("HF_TOKEN exists:", bool(HF_TOKEN))
print(
    "HF_TOKEN starts correctly:",
    HF_TOKEN.startswith("hf_")
)

# Create DuckDB connection
con = duckdb.connect()

# Recreate Hugging Face secret
con.execute("DROP SECRET IF EXISTS hf")

con.execute(
    f"""
    CREATE SECRET hf (
        TYPE huggingface,
        TOKEN '{HF_TOKEN.strip()}'
    )
    """
)

# Warehouse path
BASE = (
    "hf://datasets/FlyRank/internship-warehouse/"
    "fact_content_daily_performance/"
)

print("DuckDB connected to Hugging Face.")

HF_TOKEN exists: True
HF_TOKEN starts correctly: True
DuckDB connected to Hugging Face.


In [3]:
# Check available months using DuckDB.
# Nothing is loaded into pandas except the tiny result.

months = con.sql(f"""
SELECT
    month,
    COUNT(*) AS rows
FROM read_parquet(
    '{BASE}month=*/**/*.parquet',
    hive_partitioning = true
)
GROUP BY month
ORDER BY month
""").df()

display(months)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,month,rows
0,2025-01,1297
1,2025-02,75985
2,2025-03,167859
3,2025-04,285114
4,2025-05,349923
5,2025-06,329201
6,2025-07,469794
7,2025-08,704962
8,2025-09,845813
9,2025-10,2165471


3A — First check which months are available

In [4]:
# Check available months without loading the dataset into pandas.

months = con.sql("""
SELECT
    month,
    COUNT(*) AS rows
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
)
GROUP BY month
ORDER BY month
""").df()

display(months)

,month,rows
0,2025-01,1297
1,2025-02,75985
2,2025-03,167859
3,2025-04,285114
4,2025-05,349923
5,2025-06,329201
6,2025-07,469794
7,2025-08,704962
8,2025-09,845813
9,2025-10,2165471


3B — Build the modeling data in DuckDB

In [5]:
TRAIN_MONTH = "2026-02"
TEST_MONTH = "2026-03"

model_data = con.sql(f"""
WITH train AS (
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(gsc_impressions) AS impressions,
        SUM(gsc_clicks) AS clicks,

        SUM(gsc_clicks) * 1.0
            / NULLIF(SUM(gsc_impressions), 0) AS ctr,

        AVG(gsc_avg_position) AS avg_position

    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/'
        'fact_content_daily_performance/month={TRAIN_MONTH}/*.parquet'
    )

    WHERE gsc_data_available IS TRUE

    GROUP BY
        client_hash_id,
        content_hash_id

    HAVING SUM(gsc_impressions) > 0
),

future AS (
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(gsc_impressions) AS future_impressions,
        SUM(gsc_clicks) AS future_clicks,

        SUM(gsc_clicks) * 1.0
            / NULLIF(SUM(gsc_impressions), 0) AS future_ctr

    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/'
        'fact_content_daily_performance/month={TEST_MONTH}/*.parquet'
    )

    WHERE gsc_data_available IS TRUE

    GROUP BY
        client_hash_id,
        content_hash_id
)

SELECT
    t.client_hash_id,
    t.content_hash_id,

    t.impressions,
    t.clicks,
    t.ctr,
    t.avg_position,

    f.future_impressions,
    f.future_clicks,
    f.future_ctr,

    CASE
        WHEN f.future_ctr < t.ctr
        THEN 1
        ELSE 0
    END AS target_decline

FROM train t

INNER JOIN future f
    ON t.client_hash_id = f.client_hash_id
   AND t.content_hash_id = f.content_hash_id

WHERE t.ctr IS NOT NULL
  AND t.avg_position IS NOT NULL
  AND f.future_ctr IS NOT NULL
""").df()

print("Modeling rows:", len(model_data))
display(model_data.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Modeling rows: 134238


,client_hash_id,content_hash_id,impressions,clicks,ctr,avg_position,future_impressions,future_clicks,future_ctr,target_decline
0,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,601.0,3.0,0.004992,4.527386,1140.0,2.0,0.001754,1
1,client_73cda7b4e4f265ea,content_05597932fe4da067,207.0,1.0,0.004831,2.991600,57.0,0.0,0.000000,1
2,client_73cda7b4e4f265ea,content_905aa32a0230694e,156.0,1.0,0.006410,8.661331,149.0,0.0,0.000000,1
3,client_73cda7b4e4f265ea,content_05434271b257bb68,1567.0,7.0,0.004467,5.662970,1421.0,6.0,0.004222,1
4,client_73cda7b4e4f265ea,content_d056587ff7faca0c,2176.0,4.0,0.001838,3.732656,2770.0,16.0,0.005776,0


3C — Train Logistic Regression

In [6]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)
import numpy as np

FEATURES = [
    "impressions",
    "ctr",
    "avg_position"
]

X = model_data[FEATURES].copy()
y = model_data["target_decline"].astype(int)
groups = model_data["client_hash_id"]

# Reduce the effect of the heavy-tailed impressions distribution.
X["impressions"] = np.log1p(X["impressions"])

# Grouped split: a client is kept entirely in train or test.
gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

model = Pipeline([
    ("scaler", StandardScaler()),
    ("logistic", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42
    ))
])

model.fit(X_train, y_train)

model_probability = model.predict_proba(X_test)[:, 1]
model_prediction = (model_probability >= 0.5).astype(int)

print("Train rows:", len(X_train))
print("Test rows:", len(X_test))
print("Train clients:", groups.iloc[train_idx].nunique())
print("Test clients:", groups.iloc[test_idx].nunique())

Train rows: 88344
Test rows: 45894
Train clients: 33
Test clients: 9


3D — Evaluate the model

In [7]:
model_metrics = {
    "Model": "Logistic Regression",
    "Accuracy": accuracy_score(y_test, model_prediction),
    "Precision": precision_score(
        y_test,
        model_prediction,
        zero_division=0
    ),
    "Recall": recall_score(
        y_test,
        model_prediction,
        zero_division=0
    ),
    "F1": f1_score(
        y_test,
        model_prediction,
        zero_division=0
    ),
    "ROC_AUC": roc_auc_score(
        y_test,
        model_probability
    )
}

display(
    __import__("pandas").DataFrame([model_metrics])
)

,Model,Accuracy,Precision,Recall,F1,ROC_AUC
0,Logistic Regression,0.887611,0.710237,0.902591,0.794943,0.948492


In [8]:
# Recreate the Week-4 baseline logic on the exact model test rows.

comparison = model_data.iloc[test_idx].copy()

comparison["baseline_score"] = 0
comparison["baseline_prediction"] = 0

high_volume = comparison["impressions"] >= 100
low_ctr = comparison["ctr"] < 0.02

comparison.loc[
    high_volume & low_ctr,
    "baseline_score"
] = 2

comparison.loc[
    high_volume & low_ctr,
    "baseline_prediction"
] = 1

comparison["model_probability"] = model_probability
comparison["model_prediction"] = model_prediction

In [9]:
baseline_prediction = comparison["baseline_prediction"].astype(int)

baseline_metrics = {
    "Model": "Week-4 Baseline",
    "Accuracy": accuracy_score(
        y_test,
        baseline_prediction
    ),
    "Precision": precision_score(
        y_test,
        baseline_prediction,
        zero_division=0
    ),
    "Recall": recall_score(
        y_test,
        baseline_prediction,
        zero_division=0
    ),
    "F1": f1_score(
        y_test,
        baseline_prediction,
        zero_division=0
    )
}

comparison_table = __import__("pandas").DataFrame([
    model_metrics,
    baseline_metrics
])

display(comparison_table)

,Model,Accuracy,Precision,Recall,F1,ROC_AUC
0,Logistic Regression,0.887611,0.710237,0.902591,0.794943,0.948492
1,Week-4 Baseline,0.666427,0.403264,0.796335,0.535401,NaN


In [10]:
best_model = comparison_table.loc[
    comparison_table["F1"].idxmax(),
    "Model"
]

best_f1 = comparison_table["F1"].max()

print(
    f"Based on F1 on the same held-out test rows, "
    f"{best_model} performed better with an F1 of {best_f1:.3f}."
)

Based on F1 on the same held-out test rows, Logistic Regression performed better with an F1 of 0.795.


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [11]:
# Error analysis on the held-out test set

error_analysis = comparison.copy()

error_analysis["actual"] = y_test.to_numpy()
error_analysis["predicted"] = model_prediction

error_analysis["error_type"] = "CORRECT"

error_analysis.loc[
    (error_analysis["actual"] == 0) &
    (error_analysis["predicted"] == 1),
    "error_type"
] = "FALSE_POSITIVE"

error_analysis.loc[
    (error_analysis["actual"] == 1) &
    (error_analysis["predicted"] == 0),
    "error_type"
] = "FALSE_NEGATIVE"

print("Error counts:")
display(error_analysis["error_type"].value_counts())

print("\nError rates:")
display(
    error_analysis["error_type"]
    .value_counts(normalize=True)
    .rename("proportion")
)

Error counts:


,count
error_type,
CORRECT,40736
FALSE_POSITIVE,4079
FALSE_NEGATIVE,1079



Error rates:


,proportion
error_type,
CORRECT,0.887611
FALSE_POSITIVE,0.088879
FALSE_NEGATIVE,0.023511


In [12]:
# Inspect representative false positives and false negatives

print("Representative false positives:")
display(
    error_analysis[
        error_analysis["error_type"] == "FALSE_POSITIVE"
    ][
        [
            "client_hash_id",
            "content_hash_id",
            "impressions",
            "ctr",
            "avg_position",
            "actual",
            "predicted",
            "model_probability"
        ]
    ].head(10)
)

print("Representative false negatives:")
display(
    error_analysis[
        error_analysis["error_type"] == "FALSE_NEGATIVE"
    ][
        [
            "client_hash_id",
            "content_hash_id",
            "impressions",
            "ctr",
            "avg_position",
            "actual",
            "predicted",
            "model_probability"
        ]
    ].head(10)
)

Representative false positives:


,client_hash_id,content_hash_id,impressions,ctr,avg_position,actual,predicted,model_probability
4,client_73cda7b4e4f265ea,content_d056587ff7faca0c,2176.0,0.001838,3.732656,0,1,0.672729
6,client_73cda7b4e4f265ea,content_2662845f598544ef,316.0,0.006329,5.203637,0,1,0.868619
11,client_73cda7b4e4f265ea,content_d720dde3701523c0,152.0,0.006579,29.782737,0,1,0.840853
23,client_73cda7b4e4f265ea,content_0c29bba4e77ab325,3516.0,0.001706,4.903080,0,1,0.730084
45,client_73cda7b4e4f265ea,content_604ecfdce14c9458,1969.0,0.003047,5.104703,0,1,0.792369
59,client_73cda7b4e4f265ea,content_38b6eb3d080e23c7,2373.0,0.001264,5.099239,0,1,0.614746
64,client_73cda7b4e4f265ea,content_090e4dfc1196be61,500.0,0.004000,9.926040,0,1,0.717413
110,client_73cda7b4e4f265ea,content_29393234d4a0fea6,5876.0,0.002212,7.848893,0,1,0.840772
146,client_73cda7b4e4f265ea,content_b73a9d9e642898e6,6885.0,0.001162,3.939263,0,1,0.761411
150,client_73cda7b4e4f265ea,content_14bd3cd8b7e29e1b,4538.0,0.000220,4.804770,0,1,0.583850


Representative false negatives:


,client_hash_id,content_hash_id,impressions,ctr,avg_position,actual,predicted,model_probability
25,client_73cda7b4e4f265ea,content_17680db83b4a16c9,1298.0,0.000770,9.090911,1,0,0.446402
51,client_73cda7b4e4f265ea,content_b60788d1e0043b2e,633.0,0.001580,3.994666,1,0,0.425176
100,client_73cda7b4e4f265ea,content_5a7e030078c41c07,1031.0,0.000970,12.185560,1,0,0.438501
102,client_73cda7b4e4f265ea,content_3e17d8e139f607d9,1743.0,0.000574,16.676344,1,0,0.482760
111,client_73cda7b4e4f265ea,content_93f85df43e0ba476,435.0,0.002299,21.101358,1,0,0.485998
113,client_73cda7b4e4f265ea,content_ad51a7cec7cad5a5,1246.0,0.000803,12.402144,1,0,0.448805
151,client_73cda7b4e4f265ea,content_b7743cf568323323,678.0,0.001475,3.319659,1,0,0.421639
324,client_73cda7b4e4f265ea,content_9764760dd135142f,434.0,0.002304,10.649218,1,0,0.470046
407,client_73cda7b4e4f265ea,content_58a026f53000194e,478.0,0.002092,1.630306,1,0,0.443438
446,client_73cda7b4e4f265ea,content_087602f4b2740093,1164.0,0.000859,13.639389,1,0,0.446622


## Errors and interpretation

The Logistic Regression model performs better than the Week-4 baseline on the same held-out observations. Its F1 score is **0.795**, compared with **0.535** for the Week-4 baseline.

The model still produces both false positives and false negatives. False positives are pages predicted to decline whose later observed CTR does not decline. False negatives are pages whose later CTR declines but the model does not identify them. These errors are expected because search performance can also be affected by factors such as search intent, SERP features, seasonality, competition, and changes in search demand.

The model uses **impressions, CTR, and average position** as its main signals. Impressions are log-transformed because the distribution is heavily right-skewed. This prevents very high-volume pages from dominating the model only because of their raw scale.

The error review shows that a prediction should not automatically trigger a content refresh. A false positive can occur when low CTR is caused by search intent or SERP conditions rather than a content problem, while a false negative can occur when an external change causes later performance to decline without a strong signal in the earlier features.

Therefore, the model is useful as **decision-support** and improves on the Week-4 rule in this evaluation, but it should still be followed by human review.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.